# Weighted (Anisotropic) Laplacian Operators

## Overview

The classical Laplacian $-\Delta u = -\sum_i \partial_{x_i}^2 u$ treats all spatial directions and positions equally. In many applications — heat conduction in heterogeneous media, diffusion in biological tissue, image-adaptive smoothing — the material properties vary in space, demanding a **weighted** or **anisotropic** Laplacian that accounts for local conductivity or diffusivity.

### The weighted 1D operator

In one dimension, the weighted Laplacian is:
$$\mathcal{L}_w u = -\frac{d}{dx}\!\left(w(x)\frac{du}{dx}\right)$$

where $w(x) > 0$ is a spatially varying weight (conductivity). When $w$ is constant this reduces to $-w\,u''$.

### Finite difference discretization in 1D

Using the staggered-grid (midpoint) weights $w_{i+1/2} = (w_i + w_{i+1})/2$, the discrete weighted Laplacian reads:
$$[L_w]_{ij} = \begin{cases}
w_{i-1/2} + w_{i+1/2} & j = i \\
-w_{i-1/2} & j = i-1 \\
-w_{i+1/2} & j = i+1 \\
0 & \text{otherwise}
\end{cases}$$
scaled by $1/h^2$. This matrix is symmetric positive-definite for any positive $w$.

### The weighted 2D operator

In two dimensions:
$$\mathcal{L}_w u = -\nabla \cdot (w(x,y) \nabla u) = -\partial_x(w\,\partial_x u) - \partial_y(w\,\partial_y u)$$

The anisotropy can be extended to a full tensor weight $W(x,y) \in \mathbb{R}^{2\times 2}$, encoding directional conductivity. In the scalar case, eigenfunctions **concentrate in low-$w$ regions** (where diffusion is slow) and oscillate rapidly in high-$w$ regions.

### Applications

- **Variable conductivity heat equation**: $\partial_t u = \nabla \cdot (w \nabla u)$.
- **Image-adaptive smoothing**: $w$ computed from image gradients to preserve edges while smoothing flat regions.
- **Manifold Laplacians**: weighted graph Laplacians in spectral clustering.
- **Finite element method**: the stiffness matrix for the diffusion equation is exactly $L_w$.

### What this notebook demonstrates

1. 1D eigenfunction evolution as the weight profile $w(x)$ varies (uniform, step, sinusoidal).
2. 2D eigenmodes of a domain with spatially varying (heterogeneous) conductivity.
3. Effect of anisotropy: weight aligned along one axis produces elongated eigenmodes.

### Imports

We use `scipy.sparse` for sparse matrix construction and `scipy.sparse.linalg.eigsh` for partial eigendecomposition.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.sparse import diags, kron, eye, csr_matrix
from scipy.sparse.linalg import eigsh

### 1D weighted Laplacian assembly

We assemble the discrete weighted Laplacian for a 1D domain $[0, 1]$ with $n$ interior points and Dirichlet BC:
$$[L_w]_{ij} = \frac{1}{h^2} \begin{cases}
w_{i-1/2} + w_{i+1/2} & j = i \\
-w_{i+1/2} & j = i+1 \\
-w_{i-1/2} & j = i-1
\end{cases}$$

The midpoint conductivity $w_{i+1/2}$ is the harmonic mean of adjacent values (or arithmetic mean as an approximation).

In [2]:
def weighted_laplacian_1d(w_vals, h):
    """
    Assemble 1D weighted Laplacian on n interior points.
    w_vals: weight at interior nodes (length n)
    Boundary nodes have weight w=1 (Dirichlet: u=0)
    """
    n = len(w_vals)
    # Extend w to include boundary ghost nodes
    w_ext = np.concatenate([[1.0], w_vals, [1.0]])
    # Staggered midpoint weights
    w_half_plus = 0.5 * (w_ext[1:-1] + w_ext[2:])    # w_{i+1/2}, length n
    w_half_minus = 0.5 * (w_ext[:-2] + w_ext[1:-1])   # w_{i-1/2}, length n
    diag = (w_half_minus + w_half_plus) / h**2
    off_plus = -w_half_plus[:-1] / h**2
    off_minus = -w_half_minus[1:] / h**2
    L = diags([off_minus, diag, off_plus], [-1, 0, 1], format='csr')
    return L

n1d = 200
h1d = 1.0 / (n1d + 1)
x1d = np.linspace(h1d, 1 - h1d, n1d)

# Several weight profiles
weight_profiles = {
    'Uniform $w=1$':    np.ones(n1d),
    'Step $w$':          np.where(x1d < 0.5, 0.2, 2.0),
    'Sinusoidal $w$':    1.0 + 0.8 * np.sin(2 * np.pi * 2 * x1d),
    'Gaussian well':     0.5 + 2.0 * np.exp(-((x1d - 0.5)**2) / (2 * 0.1**2)),
}

n_modes_1d = 6
eig_results_1d = {}
for name, w in weight_profiles.items():
    L = weighted_laplacian_1d(w, h1d)
    vals, vecs = eigsh(L, k=n_modes_1d, which='SM', tol=1e-10)
    idx = np.argsort(vals)
    eig_results_1d[name] = {'vals': vals[idx], 'vecs': vecs[:, idx], 'w': w}

### Eigenfunction evolution across weight profiles

We compare the first 6 eigenfunctions for each weight profile. For the uniform weight $w = 1$, the eigenfunctions are the classical sine functions $\sin(k\pi x)$. When $w$ is non-uniform, eigenfunctions **adapt to the medium**:
- In regions of low conductivity (small $w$), oscillations are slow and eigenfunctions concentrate.
- In regions of high conductivity (large $w$), the function tends to vary faster.

This is analogous to quantum mechanics where low-potential wells trap wavefunctions.

In [3]:
fig, axes = plt.subplots(len(weight_profiles), n_modes_1d + 1, figsize=(16, 10))

for row, (name, res) in enumerate(eig_results_1d.items()):
    # First column: weight profile
    axes[row, 0].fill_between(x1d, 0, res['w'], alpha=0.4, color='steelblue')
    axes[row, 0].plot(x1d, res['w'], 'b-', lw=1.5)
    axes[row, 0].set_title('$w(x)$', fontsize=9)
    axes[row, 0].set_ylim(0, None)
    axes[row, 0].set_yticks([])
    axes[row, 0].set_ylabel(name, fontsize=8, rotation=45, ha='right')
    # Eigenfunctions
    for k in range(n_modes_1d):
        u = res['vecs'][:, k]
        u = u / np.max(np.abs(u))
        axes[row, k + 1].plot(x1d, u, 'k-', lw=1.2)
        axes[row, k + 1].axhline(0, color='gray', lw=0.5)
        axes[row, k + 1].fill_between(x1d, 0, u, alpha=0.25,
                                       where=u > 0, color='red')
        axes[row, k + 1].fill_between(x1d, 0, u, alpha=0.25,
                                       where=u < 0, color='blue')
        axes[row, k + 1].set_title(f'$u_{k+1}$\n$\\lambda={res["vals"][k]:.1f}$', fontsize=8)
        axes[row, k + 1].set_yticks([])
        axes[row, k + 1].set_xticks([])

plt.suptitle('1D weighted Laplacian eigenfunctions', fontsize=12)
plt.tight_layout()
plt.savefig('1d_eigenfunctions.png', dpi=80, bbox_inches='tight')
plt.close()

### 2D weighted Laplacian assembly

The 2D weighted Laplacian separates into $x$- and $y$-directions:
$$\mathcal{L}_w u = -\partial_x(w\partial_x u) - \partial_y(w\partial_y u)$$

For an isotropic scalar weight $w(x,y)$, we can use the same staggered-grid idea in each direction. The discrete operator is assembled as:
$$L_w = L_{w_x} \otimes I + I \otimes L_{w_y}$$

where $L_{w_x}$ uses the row-averaged weights and $L_{w_y}$ uses column-averaged weights. We study three configurations:
1. **Uniform** $w = 1$ (standard Laplacian).
2. **Heterogeneous**: a Gaussian inclusion of high conductivity.
3. **Anisotropic**: $w$ large along $x$, inducing preference for $x$-aligned eigenmodes.

In [4]:
def weighted_laplacian_2d(W, h):
    """
    W: (n, n) array of weights at interior nodes.
    Returns sparse (n^2, n^2) weighted Laplacian.
    """
    n = W.shape[0]
    assert W.shape[1] == n

    # For x-direction: average W over columns to get row weights
    # For y-direction: average W over rows to get column weights
    # More precisely: stagger along each axis using W directly

    def make_1d_L(w_row, h):
        """1D weighted Laplacian for a single row/column of weights."""
        return weighted_laplacian_1d(w_row, h)

    # Assemble block-diagonal structure: x-direction Laplacian
    # Each row i of the 2D grid contributes a 1D Laplacian with weights W[i, :]
    blocks_x = []
    for i in range(n):
        blocks_x.append(make_1d_L(W[i, :], h))

    from scipy.sparse import block_diag as sp_block_diag
    Lx = sp_block_diag(blocks_x, format='csr')

    # y-direction: Laplacian coupling rows, using column weights W[:, j]
    # This is trickier: we need to couple node (i, j) with (i±1, j)
    # Weight at interface (i+1/2, j): 0.5*(W[i,j] + W[i+1,j])
    n2 = n * n
    diag_main = np.zeros(n2)
    diag_up = np.zeros(n2 - n)
    diag_down = np.zeros(n2 - n)

    for i in range(n):
        for j in range(n):
            node = i * n + j
            # coupling to (i+1, j)
            if i < n - 1:
                w_half = 0.5 * (W[i, j] + W[i + 1, j]) / h**2
                diag_main[node] += w_half
                diag_main[node + n] += w_half
                diag_up[node] = -w_half
            # boundary term (i=0): ghost node contributes w_{-1/2} = w[0,j]
            if i == 0:
                w_half_bnd = W[0, j] / h**2
                diag_main[node] += w_half_bnd
            # boundary term (i=n-1): ghost node
            if i == n - 1:
                w_half_bnd = W[n - 1, j] / h**2
                diag_main[node] += w_half_bnd

    diag_down = diag_up.copy()

    Ly = diags([diag_down, diag_main, diag_up], [-n, 0, n],
               shape=(n2, n2), format='csr')

    return (Lx + Ly).tocsr()

n2d = 30
h2d = 1.0 / (n2d + 1)
x2 = np.linspace(h2d, 1 - h2d, n2d)
X2, Y2 = np.meshgrid(x2, x2)

W_uniform = np.ones((n2d, n2d))
W_hetero = 1.0 + 5.0 * np.exp(-((X2 - 0.5)**2 + (Y2 - 0.5)**2) / (2 * 0.12**2))
W_aniso = 1.0 + 3.0 * np.exp(-((Y2 - 0.5)**2) / (2 * 0.05**2))  # stripe along x

weight_configs_2d = {
    'Uniform':      W_uniform,
    'Heterogeneous (Gaussian bump)': W_hetero,
    'Anisotropic (horizontal stripe)': W_aniso,
}

n_modes_2d = 9
eig_results_2d = {}
for name, W in weight_configs_2d.items():
    L2 = weighted_laplacian_2d(W, h2d)
    vals, vecs = eigsh(L2, k=n_modes_2d, which='SM', tol=1e-8)
    idx = np.argsort(vals)
    eig_results_2d[name] = {'vals': vals[idx], 'vecs': vecs[:, idx], 'W': W}
    print(f'{name}: λ1={vals[idx[0]]:.2f}, λ2={vals[idx[1]]:.2f}')

Uniform: λ1=19.72, λ2=49.20
Heterogeneous (Gaussian bump): λ1=21.78, λ2=73.86
Anisotropic (horizontal stripe): λ1=25.76, λ2=62.33


### 2D weighted Laplacian eigenmodes

We display the first 9 eigenmodes for each weight configuration side by side. The weight map $w(x,y)$ is shown in the first panel of each row.

Key observations:
- **Uniform** $w$: classical modes aligned with the square grid directions.
- **Heterogeneous** (Gaussian bump): eigenfunctions distort around the high-conductivity inclusion, with modes that tend to concentrate away from the bump.
- **Anisotropic** (stripe): eigenfunctions become elongated in the direction favored by high conductivity.

In [5]:
from matplotlib.colors import TwoSlopeNorm

n_configs = len(weight_configs_2d)
fig, axes = plt.subplots(n_configs, n_modes_2d + 1, figsize=(20, 7))

for row, (name, res) in enumerate(eig_results_2d.items()):
    W = res['W']
    # Weight map
    axes[row, 0].contourf(X2, Y2, W, levels=20, cmap='YlOrRd')
    axes[row, 0].set_title('$w(x,y)$', fontsize=8)
    axes[row, 0].set_aspect('equal')
    axes[row, 0].axis('off')
    axes[row, 0].set_ylabel(name.split('(')[0].strip(), fontsize=8)
    # Eigenmodes
    for k in range(n_modes_2d):
        u_flat = res['vecs'][:, k]
        u_2d = u_flat.reshape(n2d, n2d)
        vmax = np.abs(u_2d).max() + 1e-8
        axes[row, k + 1].contourf(X2, Y2, u_2d, levels=20, cmap='RdBu_r',
                                   vmin=-vmax, vmax=vmax)
        try:
            axes[row, k + 1].contour(X2, Y2, u_2d, levels=[0], colors='k',
                                      linewidths=0.5)
        except Exception:
            pass
        axes[row, k + 1].set_title(f'$u_{k+1}$', fontsize=7)
        axes[row, k + 1].set_aspect('equal')
        axes[row, k + 1].axis('off')

plt.suptitle('2D Weighted Laplacian Eigenmodes', fontsize=12)
plt.tight_layout()
plt.savefig('2d_eigenmodes.png', dpi=70, bbox_inches='tight')
plt.close()

### Eigenvalue spectrum comparison

The eigenvalues $\lambda_k$ depend on both the domain shape (fixed square here) and the weight function $w$. A higher overall conductivity shifts all eigenvalues upward; heterogeneity breaks spectral degeneracies present in the uniform case.

We plot the first 9 eigenvalues for all three weight configurations on the same axis.

In [6]:
fig, ax = plt.subplots(figsize=(8, 5))
markers = ['o', 's', '^']
for (name, res), mk in zip(eig_results_2d.items(), markers):
    ax.plot(range(1, n_modes_2d + 1), res['vals'][:n_modes_2d],
            marker=mk, label=name, markersize=7)
ax.set_xlabel('Mode index $k$')
ax.set_ylabel('Eigenvalue $\\lambda_k$')
ax.set_title('Eigenvalue spectra: effect of weight $w(x,y)$')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('eigenvalue_compare.png', dpi=80, bbox_inches='tight')
plt.close()

### Interactive: weight profile and eigenmodes in 1D

The widget lets you choose the weight profile and mode index, and displays the corresponding eigenfunction alongside the weight function. Observe how the eigenfunction localizes in regions of low conductivity.

### Static snapshot

In [7]:
STATIC_SNAPSHOT = True
if STATIC_SNAPSHOT:
    # Show 1D eigenfunctions for all weight profiles (first 4 modes)
    n_show = 4
    fig, axes = plt.subplots(len(weight_profiles), n_show + 1, figsize=(14, 10))
    for row, (name, res) in enumerate(eig_results_1d.items()):
        axes[row, 0].fill_between(x1d, 0, res['w'], alpha=0.4, color='steelblue')
        axes[row, 0].plot(x1d, res['w'], 'b-', lw=1.5)
        axes[row, 0].set_title('$w(x)$', fontsize=9)
        axes[row, 0].set_yticks([])
        axes[row, 0].set_ylabel(name, fontsize=8)
        for k in range(n_show):
            u = res['vecs'][:, k]
            u = u / np.max(np.abs(u))
            axes[row, k + 1].plot(x1d, u, 'k-', lw=1.2)
            axes[row, k + 1].axhline(0, color='gray', lw=0.5)
            axes[row, k + 1].fill_between(x1d, 0, u, alpha=0.2,
                                           where=u > 0, color='red')
            axes[row, k + 1].fill_between(x1d, 0, u, alpha=0.2,
                                           where=u < 0, color='blue')
            axes[row, k + 1].set_title(f'$u_{k+1}$', fontsize=9)
            axes[row, k + 1].set_yticks([])
            axes[row, k + 1].set_xticks([])
    plt.suptitle('Weighted Laplacian Eigenfunctions (1D)', fontsize=12)
    plt.tight_layout()
    plt.savefig('snippet.png', dpi=100, bbox_inches='tight')
    plt.close()

## Takeaways

- The **weighted Laplacian** $\mathcal{L}_w = -\nabla\cdot(w\nabla)$ generalizes the standard Laplacian to heterogeneous or anisotropic media.
- Its discrete form uses **staggered midpoint weights**, preserving the symmetric positive-definite structure critical for numerical stability.
- Eigenfunctions **adapt to the medium**: they concentrate in low-conductivity regions and vary rapidly in high-conductivity zones.
- Breaking translational symmetry (via spatially varying $w$) splits eigenvalue degeneracies present in the uniform case.
- Applications span variable-conductivity heat equations, image-adaptive diffusion, finite-element stiffness matrices, and manifold Laplacians in machine learning.

## Bibliography

- L. C. Evans, *Partial Differential Equations*, American Mathematical Society, 2nd edition, 2010. Chapter 6.
- G. Dziuk and C. M. Elliott, *Finite element methods for surface PDEs*, Acta Numerica, 22:289–396, 2013.
- P. Perona and J. Malik, *Scale-space and edge detection using anisotropic diffusion*, IEEE PAMI, 12(7):629–639, 1990.
- M. Belkin and P. Niyogi, *Laplacian eigenmaps for dimensionality reduction and data representation*, Neural Computation, 15(6):1373–1396, 2003.
- A. Quarteroni, R. Sacco, F. Saleri, *Numerical Mathematics*, Springer, 2007. Chapter 12.